In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
files=reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
from minsearch import Index

query = "How does the agentic loop keep calling the model until it stops?"

index = Index(
text_fields=["content"],
keyword_fields=["filename"]
)
index.fit(documents)

results = index.search(query, num_results=5)
results[0]["filename"], results[:2]

('01-agentic-rag/lessons/14-agentic-loop.md',
 [{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- T

In [19]:
from openai import OpenAI
import os

INSTRUCTIONS = """
You are a course teaching assistant.
Answer only from the provided context.
If the answer is not in context, say: I don't know.
""".strip()

PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
MODEL = "gemini-3.6-flash"

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY is not set")

client = OpenAI(api_key=api_key, base_url=GEMINI_BASE_URL)

def search_lessons(query, num_results=5):
    return index.search(query, num_results=num_results)

def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append(f"FILE: {doc['filename']}")
        lines.append(doc["content"])
        lines.append("")
    return "\n".join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    return PROMPT_TEMPLATE.format(question=question, context=context)

def get_text_from_response(response):
    if hasattr(response, "output_text") and response.output_text:
        return response.output_text
    if hasattr(response, "choices") and response.choices:
        message = response.choices[0].message
        return getattr(message, "content", "") or ""
    return ""

def get_input_tokens(response):
    usage = getattr(response, "usage", None)
    if usage is None:
        return None

    input_tokens = getattr(usage, "input_tokens", None)
    if input_tokens is not None:
        return input_tokens

    prompt_tokens = getattr(usage, "prompt_tokens", None)
    if prompt_tokens is not None:
        return prompt_tokens

    if isinstance(usage, dict):
        return usage.get("input_tokens") or usage.get("prompt_tokens")

    return None

def llm_call(prompt, model=MODEL):
    input_messages = [
        {"role": "system", "content": INSTRUCTIONS},
        {"role": "user", "content": prompt},
    ]

    if hasattr(client, "responses"):
        try:
            return client.responses.create(model=model, input=input_messages)
        except Exception:
            pass

    return client.chat.completions.create(
        model=model,
        messages=input_messages,
    )

def rag(question, model=MODEL):
    sr = search_lessons(question, num_results=5)
    prompt = build_prompt(question, sr)
    response = llm_call(prompt, model=model)
    answer = get_text_from_response(response)
    input_tokens = get_input_tokens(response)
    return answer, input_tokens, sr

In [20]:
q = "How does the agentic loop keep calling the model until it stops?"
answer, input_tokens, sr = rag(q, model=MODEL)
print(answer)
print("input_tokens:", input_tokens)

Based on the provided context, the agentic loop keeps calling the model using a `while` loop (such as `while True`) and a tracking flag (e.g., `has_function_calls`):

1. **Tracks Function Calls:** At the start of each iteration, `has_function_calls` is set to `False`.
2. **Processes Model Output:** The code inspects the items returned in `response.output`. If an item is a `function_call`, the agent executes the tool, appends the result to the message history, and sets `has_function_calls = True`.
3. **Exit Condition:** At the end of the iteration, the loop checks if `has_function_calls` is `False`. If the model returned a response without any function calls (a final answer message), the loop breaks and stops calling the model.
input_tokens: 7927


In [21]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print("chunks:", len(chunks))

chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

def search_chunks(query, num_results=5):
    return chunk_index.search(query, num_results=num_results)

def rag_chunks(question, model=MODEL):
    sr = search_chunks(question, num_results=5)
    prompt = build_prompt(question, sr)
    response = llm_call(prompt, model=model)
    return get_text_from_response(response), get_input_tokens(response), sr

ans2, input_tokens2, sr2 = rag_chunks(q, model=MODEL)
print("chunked input_tokens:", input_tokens2)

chunks: 295
chunked input_tokens: 2579


In [22]:
import json

search_calls = {"count": 0}

def search_tool(query: str) -> list[dict]:
    """Search the chunk index for relevant lesson passages."""
    search_calls["count"] += 1
    docs = search_chunks(query, num_results=3)
    compact = []
    for d in docs:
        compact.append({
            "filename": d["filename"],
            "start": d.get("start"),
            "content": d["content"][:1000],
        })
    return compact

tools = [
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search the lesson chunks for relevant information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query for finding relevant lesson content",
                    }
                },
                "required": ["query"],
            },
        },
    }
]

messages = [
    {
        "role": "system",
        "content": "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.",
    },
    {
        "role": "user",
        "content": "How does the agentic loop work, and how is it different from plain RAG?",
    },
]

final_answer = None
error_text = None

for _ in range(4):
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )
    except Exception as e:
        error_text = str(e)
        break

    msg = response.choices[0].message
    assistant_message = {"role": "assistant", "content": msg.content or ""}

    if getattr(msg, "tool_calls", None):
        assistant_message["tool_calls"] = [tc.model_dump() for tc in msg.tool_calls]

    messages.append(assistant_message)

    tool_calls = getattr(msg, "tool_calls", None) or []
    if not tool_calls:
        final_answer = msg.content
        break

    for tc in tool_calls:
        if tc.function.name != "search":
            continue

        args = json.loads(tc.function.arguments or "{}")
        tool_result = search_tool(args.get("query", ""))

        messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "name": "search",
            "content": json.dumps(tool_result, ensure_ascii=True),
        })

print(final_answer)
print("search calls:", search_calls["count"])
if error_text:
    print("agent error:", error_text)
    print("If this is 429 quota, wait ~60s and run this cell again.")

None
search calls: 4
